# Recalculate Smart AND from final saved classifiers

This notebook recalculates **Smart AND** by combining the saved final victimization and perpetration predictions and compares it against the saved direct overlap classifier.

It is designed to answer one methodological question:

> Is Table 3 based on the final saved classifiers, or was it a separate common-split benchmark?

The notebook produces:
- a common held-out subset based on participant IDs;
- Smart AND predictions: `victim_pred == 1 AND perpetration_pred == 1`;
- metrics for Smart AND and direct overlap on the same adolescents;
- consistency checks for the overlap outcome;
- CSV outputs for Supplementary Table S4.


## 1. Configure file paths

Set the three prediction files below.

Each file should ideally contain:
- a participant identifier column;
- true label column;
- predicted label column;
- optional probability column.

Do **not** rely on row order unless you are completely sure all three files contain the same adolescents in the same order.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report

# ---------------------------------------------------------------------
# EDIT THESE PATHS
# ---------------------------------------------------------------------
VICTIM_PRED_PATH = Path("victim_predictions_with_probabilities.csv")
PERP_PRED_PATH = Path("perp_predictions_with_probs.csv")
OVERLAP_PRED_PATH = Path("overlap_predictions_with_probs.csv")

# Optional output folder
OUT_DIR = Path("smart_and_recalculation_outputs")
OUT_DIR.mkdir(exist_ok=True)

# ---------------------------------------------------------------------
# Column overrides.
# In your current files:
#   Victimization ID = ori_idx
#   Perpetration ID = idx_original
#   Overlap ID = idx_original
# ---------------------------------------------------------------------
VICTIM_ID_COL = "ori_idx"
PERP_ID_COL = "idx_original"
OVERLAP_ID_COL = "idx_original"

VICTIM_TRUE_COL = "real_value"
VICTIM_PRED_COL = "predicted_value"
VICTIM_PROB_COL = "probability_class1"

PERP_TRUE_COL = "y_true"
PERP_PRED_COL = "y_pred"
PERP_PROB_COL = "y_prob_pos"

OVERLAP_TRUE_COL = "y_true"
OVERLAP_PRED_COL = "y_pred"
OVERLAP_PROB_COL = "y_prob_pos"

# If no ID column exists, set this to True only if row order is known to be identical.
ALLOW_ROW_ORDER_FALLBACK = False


## 2. Helper functions

In [ ]:
def read_predictions(path: Path, name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"{name} file not found: {path.resolve()}")
    df = pd.read_csv(path)
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    print(f"Columns: {list(df.columns)}")
    return df


def find_col(df: pd.DataFrame, candidates, required=True, label="column"):
    lower_map = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if required:
        raise ValueError(
            f"Could not auto-detect {label}. Tried {candidates}. "
            f"Available columns: {list(df.columns)}"
        )
    return None


def infer_id_col(*dfs):
    candidates = [
        "id", "ID", "Id", "participant_id", "respondent_id", "case_id",
        "original_index", "orig_index", "index", "idx", "row_id",
        "Unnamed: 0", "Unnamed: 0.1"
    ]
    common_cols = set(dfs[0].columns)
    for df in dfs[1:]:
        common_cols &= set(df.columns)
    for cand in candidates:
        if cand in common_cols:
            return cand
    # case-insensitive
    common_lower = {str(c).lower(): c for c in common_cols}
    for cand in candidates:
        if cand.lower() in common_lower:
            return common_lower[cand.lower()]
    return None


def infer_true_col(df: pd.DataFrame, outcome_name: str):
    candidates = [
        f"y_true_{outcome_name}", f"true_{outcome_name}", f"{outcome_name}_true",
        f"label_{outcome_name}", f"{outcome_name}_label",
        "y_true", "true", "label", "target", "actual", "real", "y"
    ]
    return find_col(df, candidates, required=True, label=f"{outcome_name} true label")


def infer_pred_col(df: pd.DataFrame, outcome_name: str):
    candidates = [
        f"y_pred_{outcome_name}", f"pred_{outcome_name}", f"{outcome_name}_pred",
        f"prediction_{outcome_name}", f"{outcome_name}_prediction",
        "y_pred", "pred", "prediction", "predicted", "class_pred", "yhat"
    ]
    return find_col(df, candidates, required=True, label=f"{outcome_name} predicted label")


def infer_prob_col(df: pd.DataFrame, outcome_name: str):
    candidates = [
        f"prob_{outcome_name}", f"{outcome_name}_prob",
        f"proba_{outcome_name}", f"{outcome_name}_proba",
        f"probability_{outcome_name}", f"{outcome_name}_probability",
        "y_prob", "prob", "proba", "probability", "score",
        "predicted_probability", "p1", "prob_1", "class_1_prob"
    ]
    return find_col(df, candidates, required=False, label=f"{outcome_name} probability")


def force_binary(series, name):
    s = pd.Series(series).copy()
    if s.dtype == bool:
        return s.astype(int)
    # handle strings
    if s.dtype == object:
        mapping = {
            "0": 0, "1": 1,
            "false": 0, "true": 1,
            "no": 0, "yes": 1,
            "non-victim": 0, "victim": 1,
            "non-perpetrator": 0, "perpetrator": 1,
            "non-overlap": 0, "overlap": 1
        }
        s2 = s.astype(str).str.strip().str.lower().map(mapping)
        if s2.notna().all():
            return s2.astype(int)
    # numeric
    s = pd.to_numeric(s, errors="raise")
    unique = sorted(pd.Series(s.dropna().unique()).tolist())
    if not set(unique).issubset({0, 1, 0.0, 1.0}):
        raise ValueError(f"{name} is not binary. Unique values: {unique[:20]}")
    return s.astype(int)


def binary_metrics(y_true, y_pred):
    y_true = force_binary(y_true, "y_true")
    y_pred = force_binary(y_pred, "y_pred")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if pd.notna(precision) and pd.notna(recall) and (precision + recall)
        else np.nan
    )
    balanced_accuracy = (recall + specificity) / 2
    accuracy = (tp + tn) / (tp + fp + tn + fn)
    return {
        "support": int(len(y_true)),
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "recall_sensitivity": recall,
        "specificity": specificity,
        "precision_ppv": precision,
        "npv": npv,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
        "accuracy": accuracy,
    }


def metrics_table(results_dict):
    order = [
        "recall_sensitivity", "specificity", "precision_ppv", "npv",
        "balanced_accuracy", "f1", "accuracy", "TP", "FP", "TN", "FN", "support"
    ]
    df = pd.DataFrame(results_dict).loc[order]
    percent_rows = [
        "recall_sensitivity", "specificity", "precision_ppv", "npv",
        "balanced_accuracy", "f1", "accuracy"
    ]
    display = df.copy()
    display.loc[percent_rows] = display.loc[percent_rows] * 100
    return display


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


## 3. Load prediction files and detect columns

In [ ]:
victim_raw = read_predictions(VICTIM_PRED_PATH, "Victimization")
perp_raw = read_predictions(PERP_PRED_PATH, "Perpetration")
overlap_raw = read_predictions(OVERLAP_PRED_PATH, "Overlap")

# Use per-file ID columns. If all three files share the same ID column name,
# these can be set to that same name in the configuration cell.
victim_id_col = VICTIM_ID_COL
perp_id_col = PERP_ID_COL
overlap_id_col = OVERLAP_ID_COL

missing_id_cols = []
for df_name, df, col in [
    ("victim", victim_raw, victim_id_col),
    ("perp", perp_raw, perp_id_col),
    ("overlap", overlap_raw, overlap_id_col),
]:
    if col is None or col not in df.columns:
        missing_id_cols.append((df_name, col, list(df.columns)))

if missing_id_cols:
    if not ALLOW_ROW_ORDER_FALLBACK:
        raise ValueError(
            "One or more ID columns were not found. Details: "
            + str(missing_id_cols)
            + ". Either correct VICTIM_ID_COL/PERP_ID_COL/OVERLAP_ID_COL "
            + "or set ALLOW_ROW_ORDER_FALLBACK=True only if row order is identical."
        )
    warnings.warn(
        "Falling back to row_order_id. This is only valid if all files are in exactly the same order."
    )
    for df in (victim_raw, perp_raw, overlap_raw):
        df["row_order_id"] = np.arange(len(df))
    victim_id_col = perp_id_col = overlap_id_col = "row_order_id"

victim_true_col = VICTIM_TRUE_COL or infer_true_col(victim_raw, "victim")
victim_pred_col = VICTIM_PRED_COL or infer_pred_col(victim_raw, "victim")
victim_prob_col = VICTIM_PROB_COL or infer_prob_col(victim_raw, "victim")

perp_true_col = PERP_TRUE_COL or infer_true_col(perp_raw, "perp")
perp_pred_col = PERP_PRED_COL or infer_pred_col(perp_raw, "perp")
perp_prob_col = PERP_PROB_COL or infer_prob_col(perp_raw, "perp")

overlap_true_col = OVERLAP_TRUE_COL or infer_true_col(overlap_raw, "overlap")
overlap_pred_col = OVERLAP_PRED_COL or infer_pred_col(overlap_raw, "overlap")
overlap_prob_col = OVERLAP_PROB_COL or infer_prob_col(overlap_raw, "overlap")

detected_cols = {
    "victim": {"id": victim_id_col, "true": victim_true_col, "pred": victim_pred_col, "prob": victim_prob_col},
    "perp": {"id": perp_id_col, "true": perp_true_col, "pred": perp_pred_col, "prob": perp_prob_col},
    "overlap": {"id": overlap_id_col, "true": overlap_true_col, "pred": overlap_pred_col, "prob": overlap_prob_col},
}
detected_cols


## 4. Standardize columns

In [ ]:
def standardize_prediction_df(df, id_col, true_col, pred_col, prob_col, prefix):
    cols = [id_col, true_col, pred_col]
    if prob_col is not None:
        cols.append(prob_col)
    out = df[cols].copy()
    rename = {
        id_col: "id",
        true_col: f"y_true_{prefix}",
        pred_col: f"y_pred_{prefix}",
    }
    if prob_col is not None:
        rename[prob_col] = f"prob_{prefix}"
    out = out.rename(columns=rename)
    out[f"y_true_{prefix}"] = force_binary(out[f"y_true_{prefix}"], f"y_true_{prefix}")
    out[f"y_pred_{prefix}"] = force_binary(out[f"y_pred_{prefix}"], f"y_pred_{prefix}")
    return out


victim = standardize_prediction_df(
    victim_raw, victim_id_col, victim_true_col, victim_pred_col, victim_prob_col, "victim"
)
perp = standardize_prediction_df(
    perp_raw, perp_id_col, perp_true_col, perp_pred_col, perp_prob_col, "perp"
)
overlap = standardize_prediction_df(
    overlap_raw, overlap_id_col, overlap_true_col, overlap_pred_col, overlap_prob_col, "overlap"
)

print(victim.head())
print(perp.head())
print(overlap.head())


## 5. Determine common held-out adolescents

This is the key step. Smart AND and direct overlap must be evaluated on the same adolescents.

Preferred comparison:
- `common_ids = victim_test_ids ∩ perpetration_test_ids ∩ overlap_test_ids`

If the common N is much smaller than 942, report this clearly as a **common held-out subset** comparison.


In [ ]:
common_ids = sorted(set(victim["id"]) & set(perp["id"]) & set(overlap["id"]))

print(f"Victim test/evaluation N: {len(victim):,}")
print(f"Perp test/evaluation N: {len(perp):,}")
print(f"Overlap test/evaluation N: {len(overlap):,}")
print(f"Common held-out N: {len(common_ids):,}")

if len(common_ids) == 0:
    raise ValueError("No common IDs across prediction files. Check ID columns and files.")

# Save common IDs for audit
pd.Series(common_ids, name="id").to_csv(OUT_DIR / "common_heldout_ids.csv", index=False)


## 6. Merge predictions

In [ ]:
df = (
    overlap[overlap["id"].isin(common_ids)]
    .merge(victim[victim["id"].isin(common_ids)], on="id", how="inner")
    .merge(perp[perp["id"].isin(common_ids)], on="id", how="inner")
)

print(df.shape)
df.head()


## 7. Check overlap outcome consistency

This verifies whether the stored overlap label equals:

`victimization true == 1 AND perpetration true == 1`

If this is not diagonal, do not use the merged data until IDs/coding are fixed.


In [ ]:
df["y_true_overlap_from_components"] = (
    (df["y_true_victim"] == 1) & (df["y_true_perp"] == 1)
).astype(int)

ct = pd.crosstab(
    df["y_true_overlap"],
    df["y_true_overlap_from_components"],
    rownames=["stored y_true_overlap"],
    colnames=["victim_true AND perp_true"]
)
print(ct)

n_mismatch = (df["y_true_overlap"] != df["y_true_overlap_from_components"]).sum()
print(f"Overlap label mismatches: {n_mismatch}")

if n_mismatch > 0:
    warnings.warn(
        "Stored overlap outcome does not match victim_true AND perp_true for some cases. "
        "Check IDs, coding, or whether overlap was defined from a different source."
    )


## 8. Recalculate Smart AND from final predictions

In [ ]:
df["smart_and_pred"] = (
    (df["y_pred_victim"] == 1) & (df["y_pred_perp"] == 1)
).astype(int)

# Direct overlap prediction from saved final overlap model
df["direct_overlap_pred"] = df["y_pred_overlap"].astype(int)

print(pd.crosstab(df["direct_overlap_pred"], df["smart_and_pred"],
                  rownames=["Direct overlap pred"], colnames=["Smart AND pred"]))

df.to_csv(OUT_DIR / "merged_predictions_smart_and.csv", index=False)


## 9. Compare fixed final predictions

This is the main result for a **final-classifier Smart AND** comparison.


In [ ]:
smart_metrics = binary_metrics(df["y_true_overlap"], df["smart_and_pred"])
direct_metrics = binary_metrics(df["y_true_overlap"], df["direct_overlap_pred"])

comparison = metrics_table({
    "Independent overlap classifier": direct_metrics,
    "Smart AND from final classifiers": smart_metrics,
})

comparison_rounded = comparison.round(1)
comparison_rounded


In [ ]:
comparison.to_csv(OUT_DIR / "table3_smart_and_final_classifier_comparison_raw.csv")
comparison_rounded.to_csv(OUT_DIR / "table3_smart_and_final_classifier_comparison_percent.csv")

save_json(
    {
        "detected_columns": detected_cols,
        "n_victim": len(victim),
        "n_perp": len(perp),
        "n_overlap": len(overlap),
        "n_common": len(df),
        "direct_overlap_metrics": direct_metrics,
        "smart_and_metrics": smart_metrics,
        "overlap_label_mismatches": int(n_mismatch),
    },
    OUT_DIR / "smart_and_recalculation_report.json"
)

print(f"Outputs saved to: {OUT_DIR.resolve()}")


## 10. Optional: direct overlap threshold matched to Smart AND recall

Use this only as a sensitivity analysis.  
The primary reproducibility comparison should use the saved final predictions.

This section requires an overlap probability column.


In [ ]:
if "prob_overlap" not in df.columns:
    print("No overlap probability column detected; skipping threshold-matching analysis.")
else:
    target_recall = smart_metrics["recall_sensitivity"]
    probs = pd.to_numeric(df["prob_overlap"], errors="raise")
    y_true = df["y_true_overlap"]

    thresholds = np.unique(np.r_[np.linspace(0, 1, 1001), probs.unique()])
    rows = []
    for thr in thresholds:
        pred = (probs >= thr).astype(int)
        m = binary_metrics(y_true, pred)
        rows.append({"threshold": thr, **m})
    thr_df = pd.DataFrame(rows)

    # Find closest recall to Smart AND recall; tie-breaker = higher balanced accuracy
    thr_df["recall_distance"] = (thr_df["recall_sensitivity"] - target_recall).abs()
    best = (
        thr_df.sort_values(["recall_distance", "balanced_accuracy"], ascending=[True, False])
        .iloc[0]
    )

    print("Smart AND recall:", target_recall)
    print("Closest direct-overlap threshold:")
    print(best)

    matched_pred = (probs >= best["threshold"]).astype(int)
    matched_metrics = binary_metrics(y_true, matched_pred)

    matched_comparison = metrics_table({
        "Independent overlap classifier - recall matched": matched_metrics,
        "Smart AND from final classifiers": smart_metrics,
    }).round(1)

    display(matched_comparison)

    thr_df.to_csv(OUT_DIR / "overlap_threshold_scan_to_match_smart_and.csv", index=False)
    matched_comparison.to_csv(OUT_DIR / "table3_optional_recall_matched_comparison.csv")


## 11. Suggested reporting language

Use one of these depending on the result.

### If common N = 942

> Smart AND was recalculated by combining the saved final victimization and perpetration classifiers and was evaluated on the same held-out test partition as the independent overlap classifier (n = 942).

### If common N < 942

> Smart AND was recalculated by combining the saved final victimization and perpetration classifiers and was evaluated on the common held-out subset of adolescents who were not used to train any of the three final models (n = XX).

### If row-order fallback was used

Do **not** use the result in the manuscript unless you can verify that row order corresponds to the same participants across files. Prefer regenerating prediction files with participant IDs.
